__!!! Activate smplx/.conda environment before running this notebook.__  
Won't run on Headless server (e.g. HPC) because it requires GUI.

### Initialization

In [ ]:
# Import libraries
import sys
import torch
import smplx
import trimesh
import pyrender
import numpy as np

from torch.utils.data import DataLoader
from datasets import HDF5Dataset

np.set_printoptions(threshold=sys.maxsize, precision=3, suppress=True)

# Check if CUDA is available
is_cuda_available = torch.cuda.is_available()
device = torch.device("cuda" if is_cuda_available else "cpu")

# Print device information
print(f"Device (CUDA/CPU):  {device}")
if is_cuda_available:
    print(f"GPU Name:           {torch.cuda.get_device_name(0)}")
    print(f"Device Count:       {torch.cuda.device_count()}")
    print(f"Current Device:     {torch.cuda.current_device()}")
else:
    print("CUDA is not available, using CPU.")

# Paths to SMPL models & hdf5 dataset
hdf5_file_path = '/home/nadeemshah/scratch/data/pre_processed/preprocessed_mod1_add_noise_0__include_weight_height_False__omit_contact_sobel_False__use_hover_False__mod_1__normalize_per_image_True.hdf5'
# hdf5_file_path = '/home/nashah/scratch/data/pre_processed/preprocessed_mod1_float32_add_noise_0__include_weight_height_False__omit_contact_sobel_False__use_hover_False__mod_1__normalize_per_image_True.hdf5'

smpl_feml_model_path_v1_0 = 'smpl/models/basicModel_f_lbs_10_207_0_v1.0.0.pkl'	# v1.0.0 has only 10 shape coefficients
smpl_male_model_path_v1_0 = 'smpl/models/basicmodel_m_lbs_10_207_0_v1.0.0.pkl'	# v1.0.0 has only 10 shape coefficients
smpl_feml_model_path_v1_1 = 'smpl/models/basicmodel_f_lbs_10_207_0_v1.1.0.pkl'	# v1.1.0 has 300 shape coefficients
smpl_male_model_path_v1_1 = 'smpl/models/basicmodel_m_lbs_10_207_0_v1.1.0.pkl'	# v1.1.0 has 300 shape coefficients
smpl_neut_model_path_v1_1 = 'smpl/models/basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl'	# neutral is only available in v1.1.0

# Load SMPL models
model = smplx.SMPL(smpl_feml_model_path_v1_0)
# model = smplx.SMPL(smpl_male_model_path_v1_0)
# model = smplx.SMPL(smpl_feml_model_path_v1_1)
# model = smplx.SMPL(smpl_male_model_path_v1_1)
# model = smplx.SMPL(smpl_neut_model_path_v1_1)

# DataLoader setup
batch_size = 1
train_dataset = HDF5Dataset(hdf5_file_path=hdf5_file_path, split='train')
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

### Function

In [ ]:
def visualize_smpl_3d(body_pose, global_orient, betas, transl):
	# Forward pass through the SMPL model
	output = model(body_pose=body_pose, global_orient=global_orient, betas=betas, transl=transl)
	vertices = output.vertices.detach().cpu().numpy().squeeze()  # (6890, 3)
	faces = model.faces  # (13776, 3)

	# Create Trimesh object
	mesh_trimesh = trimesh.Trimesh(vertices, faces, process=False)

	# Convert to pyrender.Mesh
	mesh = pyrender.Mesh.from_trimesh(mesh_trimesh)

	# Create scene
	scene = pyrender.Scene()
	scene.add(mesh)

	# Define a camera looking straight at the model from the front (Z+ axis)
	camera = pyrender.PerspectiveCamera(yfov=np.pi / 3.0)

	# Camera pose (eye position: [0, 0, 2] → 2m in front of model)
	camera_pose = np.array([
		[1.0, 0.0,  0.0,  0.0],   # X-axis
		[0.0, 1.0,  0.0,  0.0],   # Y-axis (shift up by 0 meter)
		[0.0, 0.0,  1.0,  2.0],   # Z-axis (2m away in front)
		[0.0, 0.0,  0.0,  1.0]
	])

	scene.add(camera, pose=camera_pose)

	# Add light for visibility
	light = pyrender.DirectionalLight(color=np.ones(3), intensity=2.0)
	scene.add(light, pose=camera_pose)

	# Viewer without default rotation control
	pyrender.Viewer(scene, use_raymond_lighting=True, run_in_thread=False, viewport_size=(800, 600), use_perspective_camera=True)

### A Random Subject from PressureNet Data

In [ ]:
for batch_index, (inputs, true_labels) in enumerate(train_loader, start=1):
	print(f"Processing Batch: {batch_index}/{len(train_loader)}")

	is_female = true_labels[:, 157].bool()

	# Extract SMPL parameters from true_labels
	betas           = true_labels[:, 72:82]     # Shape: (batch_size, 10)
	global_orient   = true_labels[:, 82:85]     # Shape: (batch_size, 3)
	body_pose       = true_labels[:, 85:154]    # Shape: (batch_size, 69)
	transl          = true_labels[:, 154:157]   # Shape: (batch_size, 3)

	# print(f"Batch Size:	{inputs.shape[0]}")
	print(f"Is Female:	{is_female}")
	print(f"Betas:		{np.array(betas)}")
	print(f"Global Orient:	{np.array(global_orient)}")
	print(f"Body Pose:	{np.array(body_pose).shape}")
	print(f"Transl:		{np.array(transl)}")

	break

visualize_smpl_3d(body_pose, global_orient, betas, transl)

### Rotate by 45 degrees around each axis

In [ ]:
# Define a custom pose (23 joints * 3D axis-angle = 69 values)
pose = torch.zeros(1, 69)

pi = np.pi/4	# pi/2=90, pi=180 degrees, pi/4=45 degrees
# Define a global orientation (pelvis rotation)
global_orient = torch.tensor([[0, 0, 0]], dtype=torch.float32)

# Define body shape parameters (identity variation)
betas = torch.tensor([[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], dtype=torch.float32)

# Define translation (position in space)
transl = torch.tensor([[0.0, 0.0, 0.0]], dtype=torch.float32)

visualize_smpl_3d(pose, global_orient, betas, transl)

In [ ]:
global_orient = torch.tensor([[pi, 0, 0]], dtype=torch.float32)
visualize_smpl_3d(pose, global_orient, betas, transl)

In [ ]:
global_orient = torch.tensor([[0, pi, 0]], dtype=torch.float32)
visualize_smpl_3d(pose, global_orient, betas, transl)

In [ ]:
global_orient = torch.tensor([[0, 0, pi]], dtype=torch.float32)
visualize_smpl_3d(pose, global_orient, betas, transl)

### User-defined Parameters

##### 1️⃣ Full T-Pose (Arms Extended)  
👉 Good for baseline comparison

In [ ]:
# Define a custom pose (23 joints * 3D axis-angle = 69 values)
pose = torch.zeros(1, 69)

# Define a global orientation (pelvis rotation)
global_orient = torch.tensor([[0, 0, 0]], dtype=torch.float32)

# Define body shape parameters (identity variation)
betas = torch.tensor([[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], dtype=torch.float32)

# Define translation (position in space)
transl = torch.tensor([[0.0, 0.0, 0.0]], dtype=torch.float32)

visualize_smpl_3d(pose, global_orient, betas, transl)

##### 2️⃣ Right Arm Raised (Like a Wave)
👉 Shows effect of modifying specific joints

In [ ]:
angle_45 = 45 * (np.pi / 180)
angle_20 = 20 * (np.pi / 180)
angle = 90 * (np.pi / 180)

pose = torch.tensor([
    0, 0, 0,  # left hip
    0.0, 0.0, 0.0,  # right hip
    0.0, 0.0, 0.0,  # spine 1
    0.0, 0.0, 0.0,	# left knee
    *([0] * 54),  # Keep remaining joints at default
    angle, 0.0, 0.0,  # right hand
], dtype=torch.float32).reshape(1, 69)

visualize_smpl_3d(pose, global_orient, betas, transl)

##### 3️⃣ Slight Forward Bend (Like Bowing)
👉 Demonstrates full-body pose change

In [ ]:
pose = torch.tensor([
    0.5, 0.0, 0.0,  # Leaning torso forward
    *([0] * 66)
], dtype=torch.float32).reshape(1, 69)

visualize_smpl_3d(pose, global_orient, betas, transl)

##### 4️⃣ Sitting Pose (Bent Knees)
👉 Good for showcasing lower body changes

In [ ]:
pose = torch.tensor([
    0.0, 0.0, 0.0,  # Joint 1
    0.0, 0.0, 0.0,  # Joint 2
    0.0, 0.0, 0.0,  # Joint 3
    *([0] * 24),  # Keep upper body neutral
    1.5, 0.0, 0.0,  # Right knee bent
    1.5, 0.0, 0.0,  # Left knee bent
    *([0] * 30)
], dtype=torch.float32).reshape(1, 69)

visualize_smpl_3d(pose, global_orient, betas, transl)

##### 5️⃣ Rotating the Entire Body (Global Orientation)
👉 Shows how the whole model rotates

In [ ]:
global_orient = torch.tensor([[0.0, 1.5, 0.0]], dtype=torch.float32)  # Rotate around Y-axis (turn left)
visualize_smpl_3d(pose, global_orient, betas, transl)

##### 6️⃣a. Making the Body Fat (Shape Change)
👉 Good for showing how body shapes affect visualization

In [ ]:
betas = torch.tensor([[3.0] * 10], dtype=torch.float32)  # Max positive shape params
visualize_smpl_3d(pose, global_orient, betas, transl)

##### 6️⃣b. Making the Body Thin (Shape Change)
👉 Good for showing how body shapes affect visualization

In [ ]:
betas = torch.tensor([[-3.0] * 10], dtype=torch.float32)  # Max negative shape params
visualize_smpl_3d(pose, global_orient, betas, transl)